# 03 — Cortex Agent

This notebook creates:
1. **Semantic View** — business-friendly model over the GOLD layer
2. **Cortex Agent** — AI-powered fleet intelligence assistant


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;
USE WAREHOUSE HYDRAB_HOL_WH;

## Semantic View

The Semantic View defines a business-friendly model for Cortex Analyst.

In [ ]:
-- Create Semantic View (uses session variable for database name)
EXECUTE IMMEDIATE $$
DECLARE
  db_name VARCHAR DEFAULT 'HYDRAB_HOL_' || CURRENT_USER();
  stmt VARCHAR;
BEGIN
  stmt := '
CREATE OR REPLACE CORTEX SEMANTIC VIEW GOLD.FLEET_SEMANTIC
AS SEMANTIC MODEL $$$
name: fleet_analytics
description: HydraB electric bus fleet analytics - vehicles, telemetry, defects

tables:
  - name: vehicles
    base_table:
      database: ' || db_name || '
      schema: GOLD
      table: DIM_VEHICLE
    dimensions:
      - name: vin
        expr: vin
        description: Vehicle Identification Number
      - name: model
        expr: model
        description: Bus model name
      - name: customer
        expr: customer
        description: Transit operator
      - name: depot
        expr: depot
        description: Home depot location
      - name: status
        expr: status
        description: Vehicle status
    metrics:
      - name: total_vehicles
        expr: COUNT(DISTINCT vin)
        description: Total number of vehicles

  - name: telemetry
    base_table:
      database: ' || db_name || '
      schema: GOLD
      table: FCT_LATEST_TELEMETRY
    dimensions:
      - name: vin
        expr: vin
      - name: event_time
        expr: event_time
    metrics:
      - name: avg_soc
        expr: AVG(battery_soc)
        description: Average battery state of charge
      - name: avg_speed
        expr: AVG(speed_kmh)
        description: Average speed in km/h
      - name: low_battery_count
        expr: COUNT_IF(battery_soc < 20)
        description: Vehicles with SOC below 20 percent

  - name: defects
    base_table:
      database: ' || db_name || '
      schema: GOLD
      table: FCT_DEFECT
    dimensions:
      - name: defect_type
        expr: defect_type
      - name: severity
        expr: severity
    metrics:
      - name: total_defects
        expr: COUNT(*)
        description: Total defect count
      - name: avg_repair_cost
        expr: AVG(repair_cost)
        description: Average repair cost in GBP
$$$;';
  EXECUTE IMMEDIATE stmt;
END;
$$;

## Cortex Agent

The Agent combines structured analytics (via Semantic View) with natural language Q&A.

In [ ]:
CREATE OR REPLACE CORTEX AGENT GOLD.FLEET_AGENT
  COMMENT = 'HydraB Fleet Intelligence Agent'
AS
  USING (
    CORTEX_ANALYST(
      SEMANTIC_VIEW => 'GOLD.FLEET_SEMANTIC'
    )
  );

In [ ]:
-- Test the agent
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'GOLD.FLEET_AGENT',
  'Which bus models have the most defects and what is their average repair cost?'
) AS agent_response;

## Gold Layer Complete!

You now have:
- Auto-refreshing Dynamic Tables in SILVER
- Star schema views in GOLD
- A Semantic View for natural language analytics
- A Cortex Agent you can ask questions

**Next:** Open `04_deploy_dashboard.ipynb` to deploy the React dashboard.